# Pueblo, Colorado — LVT Policy Analysis Template

This notebook sets up a city-level Land Value Tax (LVT) modeling workflow for **Pueblo, Colorado**.

> Status: **Template ready for data-source confirmation.**
>
> Before running this end-to-end, confirm the ArcGIS parcel layer URL, layer ID, levy scope (city-only vs full stack), and exemption fields.


## Stage A/B policy assumptions (to confirm)

Following `LVT_MODELING_GUIDE.md`, this notebook defaults to preserving Pueblo's current property tax structure except for the land/building rate shift.

- Policy scope: **City levy** (default assumption; update if needed)
- Reform type: **Split-rate LVT**
- Keep existing exemptions/abatements: **Yes**
- Keep post-tax credits/rollbacks: **Yes** (if present in source)
- Intentional departures from current law: **None unless explicitly added below**


In [ ]:
import math
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import requests
from shapely.geometry import Polygon

from lvt_utils import (
    calculate_current_tax,
    model_split_rate_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
)
from viz import create_spokane_property_category_chart, create_threshold_change_chart


In [ ]:
# Configuration
REPO_ROOT = Path(os.getcwd()).resolve().parent if Path(os.getcwd()).name == "examples" else Path(os.getcwd()).resolve()
data_dir = REPO_ROOT / "examples" / "data" / "pueblo"
data_dir.mkdir(parents=True, exist_ok=True)

# Pueblo County parcel endpoint from examples/skills/pueblo/FETCH.md
BASE_URL = "https://services1.arcgis.com/IL17xsvNU5Bmw3RY/arcgis/rest/services"
DATASET_NAME = "County_Parcels"
LAYER_ID = 0
ENDPOINT = f"{BASE_URL}/{DATASET_NAME}/FeatureServer/{LAYER_ID}/query"
WHERE_CLAUSE = "1=1"
PAGE_SIZE = 2_000
OUT_SR = 4326

# Pull only fields validated in Pueblo skill doc
OUT_FIELDS = [
    "PAR_TXT", "PAR_NUM", "Fips",
    "Owner", "OwnerOverflow", "SubOwner1", "SubOwner2",
    "OwnerStreetAddress", "OwnerCity", "OwnerState", "OwnerZip", "OwnerCountry",
    "TaxDistrict", "TaxExempt", "SeniorExemption", "Neighborhood", "Subdivision",
    "Zoning", "LegalDescription", "MobileHomePresent",
    "LandAssessedValue", "LandActualValue",
    "ImprovementsAssessedValue", "ImprovementsActualValue",
    "AssessorURL", "LevyURL", "ZoningURL",
    "electric", "gas", "water", "telecom", "Fire", "CSEPP",
    "created_date", "last_edited_date",
]

# Optional manual overrides for districts where levy is known but not parsable from URL
TAX_DISTRICT_TO_MILLAGE = {
    # "PUEBLO SCHOOL 60": 84.123,
}

# Baseline policy assumption (change if policy discussion specifies otherwise)
LAND_IMPROVEMENT_RATIO = 2.0


## Step 1: Load parcel data from Pueblo ArcGIS endpoint


In [ ]:
cache_path = data_dir / "pueblo_parcels_raw.parquet"

def fetch_pueblo_parcels(endpoint: str, out_fields: list[str], where_clause: str = "1=1", page_size: int = 2000) -> gpd.GeoDataFrame:
    count_params = {
        "f": "json",
        "where": where_clause,
        "returnCountOnly": "true",
    }
    count_resp = requests.get(endpoint, params=count_params, timeout=120)
    count_resp.raise_for_status()
    total_records = count_resp.json().get("count", 0)
    print(f"Total records reported by service: {total_records:,}")

    all_rows = []
    offset = 0
    while offset < total_records:
        params = {
            "f": "json",
            "where": where_clause,
            "outFields": ",".join(out_fields),
            "returnGeometry": "true",
            "resultOffset": offset,
            "resultRecordCount": page_size,
            "outSR": OUT_SR,
        }
        resp = requests.get(endpoint, params=params, timeout=180)
        resp.raise_for_status()
        payload = resp.json()
        features = payload.get("features", [])
        if not features:
            break

        for feature in features:
            attrs = feature.get("attributes", {})
            geom = feature.get("geometry") or {}
            rings = geom.get("rings")
            if rings:
                attrs["geometry"] = Polygon(rings[0])
                all_rows.append(attrs)

        print(f"Fetched {offset:,}–{offset + len(features):,} / {total_records:,}")
        if len(features) < page_size:
            break
        offset += len(features)

    gdf = gpd.GeoDataFrame(all_rows, geometry="geometry", crs="EPSG:4326")
    return gdf

if cache_path.exists():
    gdf = gpd.read_parquet(cache_path)
    print(f"Loaded cached parcels: {len(gdf):,}")
else:
    gdf = fetch_pueblo_parcels(
        endpoint=ENDPOINT,
        out_fields=OUT_FIELDS,
        where_clause=WHERE_CLAUSE,
        page_size=PAGE_SIZE,
    )
    gdf.to_parquet(cache_path, index=False)
    print(f"Fetched and cached parcels: {len(gdf):,}")

print(gdf.shape)
print(gdf.columns.tolist()[:50])


## Step 2: Column mapping (configured from Pueblo skill reference)


In [ ]:
def parse_millage_from_levy_url(value: str) -> float:
    if pd.isna(value):
        return np.nan
    text = str(value)

    # Handles common patterns like levy=84.123 or millage=84.123 in URL query strings
    match = re.search(r"(?:levy|millage|mill)[=:/%20]+([0-9]+(?:\.[0-9]+)?)", text, flags=re.IGNORECASE)
    if match:
        return float(match.group(1))

    # Fallback: take first plausible decimal millage number in the URL/path text
    generic_match = re.search(r"([0-9]{1,3}\.[0-9]{1,4})", text)
    if generic_match:
        return float(generic_match.group(1))

    return np.nan

# Pueblo source field mapping (from examples/skills/pueblo/FETCH.md)
COL_PARCEL_ID = "PAR_TXT"
COL_CITY_NAME = None  # countywide layer; no guaranteed situs-city field in skill spec
COL_LAND_VALUE = "LandAssessedValue"
COL_IMP_VALUE = "ImprovementsAssessedValue"
COL_TOTAL_VALUE = "TotalAssessedValue"
COL_MILLAGE = "mill_levy"
COL_EXEMPT_AMT = None
COL_EXEMPT_FLAG = "is_tax_exempt"
COL_USE = "Zoning"

# Required derived fields
gdf["TotalAssessedValue"] = (
    pd.to_numeric(gdf["LandAssessedValue"], errors="coerce").fillna(0)
    + pd.to_numeric(gdf["ImprovementsAssessedValue"], errors="coerce").fillna(0)
)
gdf["TotalActualValue"] = (
    pd.to_numeric(gdf["LandActualValue"], errors="coerce").fillna(0)
    + pd.to_numeric(gdf["ImprovementsActualValue"], errors="coerce").fillna(0)
)

gdf["is_tax_exempt"] = gdf.get("TaxExempt", "").astype(str).str.strip().str.lower().isin(["y", "yes", "true", "1"])
gdf["mill_levy"] = gdf.get("LevyURL", pd.Series(index=gdf.index, dtype="object")).apply(parse_millage_from_levy_url)

if TAX_DISTRICT_TO_MILLAGE:
    gdf["mill_levy"] = gdf["mill_levy"].fillna(gdf["TaxDistrict"].map(TAX_DISTRICT_TO_MILLAGE))

required_cols = [COL_PARCEL_ID, COL_LAND_VALUE, COL_IMP_VALUE, COL_TOTAL_VALUE, COL_USE, "TaxDistrict", "LevyURL"]
missing = [c for c in required_cols if c not in gdf.columns]
if missing:
    raise ValueError(f"Missing required Pueblo fields: {missing}")

print("Mapped columns and derived totals are ready.")
print(gdf[[COL_PARCEL_ID, COL_LAND_VALUE, COL_IMP_VALUE, COL_TOTAL_VALUE, COL_MILLAGE, COL_EXEMPT_FLAG]].head(3))
print("Mill levy null share:", gdf[COL_MILLAGE].isna().mean().round(4))


## Step 3: Filter to Pueblo city parcels

If source is county-wide, filter to parcels inside city limits.


In [ ]:
df = gdf.copy()

if COL_CITY_NAME is not None and COL_CITY_NAME in df.columns:
    city_mask = df[COL_CITY_NAME].astype(str).str.upper().str.contains("PUEBLO", na=False)
    df = df[city_mask].copy()
    print(f"Rows after city filter: {len(df):,}")
else:
    print("No city column configured; keeping county-wide records for now.")

# Keep only taxable parcels by default for baseline tax modeling
if COL_EXEMPT_FLAG in df.columns:
    before = len(df)
    df = df[~df[COL_EXEMPT_FLAG]].copy()
    print(f"Rows after TaxExempt filter: {len(df):,} (removed {before - len(df):,})")


## Step 4: Normalize numeric columns and exemptions


In [ ]:
for col in [COL_LAND_VALUE, COL_IMP_VALUE, COL_TOTAL_VALUE, COL_MILLAGE]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

if COL_EXEMPT_AMT is not None and COL_EXEMPT_AMT in df.columns:
    df[COL_EXEMPT_AMT] = pd.to_numeric(df[COL_EXEMPT_AMT], errors="coerce").fillna(0)

# Keep rows with modeled tax inputs available
before_numeric_filter = len(df)
df = df.dropna(subset=[COL_TOTAL_VALUE, COL_MILLAGE]).copy()
print(f"Rows after dropping null total/millage: {len(df):,} (removed {before_numeric_filter - len(df):,})")

exemption_flag_col = COL_EXEMPT_FLAG if COL_EXEMPT_FLAG in df.columns else None


## Step 5: Property category mapping

Update `category_map` to match Pueblo's assessor use codes/descriptions.


In [ ]:
category_map = {
    "SFR": "Single Family Residential",
    "SINGLE": "Single Family Residential",
    "R-": "Single Family Residential",
    "RM": "Small Multi-Family (2-4 units)",
    "MULTI": "Large Multi-Family (5+ units)",
    "APART": "Large Multi-Family (5+ units)",
    "VAC": "Vacant Land",
    "COMM": "Commercial",
    "IND": "Industrial",
}

if COL_USE in df.columns:
    use_series = df[COL_USE].astype(str).str.upper()
    df["PROPERTY_CATEGORY"] = "Other"
    for key, value in category_map.items():
        df.loc[use_series.str.contains(key, na=False), "PROPERTY_CATEGORY"] = value
else:
    df["PROPERTY_CATEGORY"] = "Other"

print(df["PROPERTY_CATEGORY"].value_counts().head(15))


## Step 6: Calculate current tax baseline


In [ ]:
if COL_TOTAL_VALUE not in df.columns or COL_MILLAGE not in df.columns:
    raise ValueError("Update column mapping before calculating current tax.")

current_revenue, _, df = calculate_current_tax(
    df=df,
    tax_value_col=COL_TOTAL_VALUE,
    millage_rate_col=COL_MILLAGE,
    exemption_col=COL_EXEMPT_AMT,
    exemption_flag_col=exemption_flag_col,
    land_value_col=COL_LAND_VALUE if COL_LAND_VALUE in df.columns else None,
    improvement_value_col=COL_IMP_VALUE if COL_IMP_VALUE in df.columns else None,
)

print(f"Current modeled revenue: ${current_revenue:,.0f}")
print(df[["current_tax", "current_taxable_value"]].describe())
print("WARNING: If many mill_levy values are null, provide TAX_DISTRICT_TO_MILLAGE overrides.")


## Step 7: Run split-rate LVT scenario (default 2:1)


In [ ]:
land_millage, imp_millage, new_revenue, df = model_split_rate_tax(
    df=df,
    land_value_col=COL_LAND_VALUE,
    improvement_value_col=COL_IMP_VALUE,
    current_revenue=current_revenue,
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
    exemption_col=COL_EXEMPT_AMT,
    exemption_flag_col=exemption_flag_col,
)

print(f"Land millage: {land_millage:.4f}")
print(f"Improvement millage: {imp_millage:.4f}")
print(f"New modeled revenue: ${new_revenue:,.0f}")


## Step 8: Summaries and charts


In [ ]:
category_summary = calculate_category_tax_summary(
    df=df,
    category_col="PROPERTY_CATEGORY",
    current_tax_col="current_tax",
    new_tax_col="new_tax",
    pct_threshold=10.0,
)

print_category_tax_summary(category_summary, title="Pueblo 2:1 LVT — Category Summary")

create_spokane_property_category_chart(
    summary_df=category_summary,
    title="Pueblo, CO 2:1 Split-Rate Tax Impact by Property Category",
    min_count=10,
)

create_threshold_change_chart(
    summary_df=category_summary,
    title="Pueblo, CO Share of Parcels with Tax Changes Greater than 10%",
    threshold=10.0,
    min_count=10,
)


## Step 9: Save modeled output


In [ ]:
out_path = data_dir / "pueblo_modeled_2to1.parquet"
df.to_parquet(out_path, index=False)
print(f"Saved modeled output: {out_path}")


## Notes for completion

- Confirm Pueblo data source URL + layer metadata.
- Finalize field mapping and exemption logic based on Pueblo assessor schema.
- Validate current modeled revenue against an official Pueblo levy benchmark.
- Optionally add Census equity analysis once parcel geography is validated.
